# 04. Балльная скор-карта и рейтинговая шкала

ТЗ §2.3.5–2.3.6: второй, **объясняющий слой** — каждому значимому признаку соответствует вклад в баллах, а PD отображается на рейтинговую шкалу из ≥ 7 градаций.

Логика: итоговая PD = `sigmoid(a·(β₀ + Σ βⱼ·WOEⱼ) + b)` (логрегрессия + калибровка Платта). Это линейно по WOE, поэтому раскладывается на аддитивные вклады признаков и переводится в баллы:

`Score = Base − Factor · (logit(PD) − logit(PD_ref))`, `Factor = PDO / ln 2`

Принято: **600 баллов ↔ PD 5%**, **каждые 50 баллов удваивают шансы «хороший/плохой»** (PDO = 50). Баллы выше = риск ниже.

In [1]:
import sys, json, warnings, joblib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "risk-scoring")); sys.path.insert(0, str(ROOT))
from riskscore.features import compute_features, FEATURE_META
from riskscore import metrics as M
MODELS, IMG = ROOT / "risk-scoring/models", ROOT / "docs/ml/img"
STR = {"inn": str, "ogrn": str, "okved": str, "region_code": str}
raw = pd.read_csv(ROOT / "data/generated/applications_raw.csv", dtype=STR)
split = pd.read_csv(ROOT / "data/generated/split.csv").split.to_numpy()
y = raw.default_flag.astype(int).to_numpy()
model = joblib.load(MODELS / "risk_model_woe_logit_v1.joblib")
pipe = model.pipeline
clip, enc, sel, lr = (pipe.named_steps[k] for k in ("clip", "woe", "select", "lr"))
cols = sel.columns
a, b = float(model.calibrator.coef_[0, 0]), float(model.calibrator.intercept_[0])
beta, b0 = lr.coef_[0], float(lr.intercept_[0])
print("модель:", model.version, "| признаков в скор-карте:", len(cols))
print(f"калибровка Платта: a={a:.4f}, b={b:.4f} | интерсепт LR={b0:.4f}")

модель: v1.0.0-woe-logit | признаков в скор-карте: 25
калибровка Платта: a=1.1074, b=0.1742 | интерсепт LR=-2.1168


## 1. Баллы по корзинам каждого признака

In [2]:
PDO, BASE_SCORE, PD_REF = 50.0, 600.0, 0.05
FACTOR = PDO / np.log(2)
logit = lambda p: np.log(p / (1 - p))
base_pts = BASE_SCORE + FACTOR * logit(PD_REF) - FACTOR * (a * b0 + b)

card = {}
for j, c in enumerate(cols):
    t = enc.bins_[c]["table"].copy()
    pts = -FACTOR * a * beta[j] * t["woe"].to_numpy()
    if enc.bins_[c]["kind"] == "num" and t["bin"].iloc[-1] != "__MISSING__":      # корзины «нет данных» не было -> 0 баллов
        t = pd.concat([t, pd.DataFrame([dict(bin="__MISSING__", n=0, bad=0, bad_rate=np.nan, woe=0.0, iv=0.0)])], ignore_index=True)
        pts = np.append(pts, 0.0)
    share = t["n"] / t["n"].sum()
    ref = float((share * pts).sum())
    card[c] = dict(label=FEATURE_META[c]["label"], kind=enc.bins_[c]["kind"], beta=float(beta[j]), bins=list(t["bin"]),
                   n=[int(x) for x in t["n"]], bad_rate=[None if np.isnan(x) else float(x) for x in t["bad_rate"]],
                   points=[float(x) for x in pts], ref_points=ref)
rng_pts = pd.Series({c: max(v["points"]) - min(v["points"]) for c, v in card.items()}).sort_values(ascending=False)
print(f"Base (константа) = {base_pts:.1f} баллов")
print("размах баллов по признакам (вес признака в скор-карте):"); print(rng_pts.round(1).to_string())

Base (константа) = 544.1 баллов
размах баллов по признакам (вес признака в скор-карте):
payment_burden            85.9
loss_quarters             68.2
equity_ratio              53.4
overdue_30_cnt            48.8
revenue_growth_yoy        35.4
overdue_60_cnt            32.4
liabilities_to_revenue    28.6
current_ratio             25.0
gross_margin              25.0
current_overdue_flag      23.1
inquiries_3m              21.5
inquiries_12m             20.2
overdue_90_cnt            19.7
negative_equity           19.4
company_age_months        18.1
balance_to_turnover       16.4
revenue_cv                15.7
turnover_cv                8.7
debit_credit_ratio         5.6
turnover_to_revenue        5.4
account_blocked            4.8
history_months             3.6
turnover_avg_log           3.6
open_loans                 2.7
tax_debt_ratio             1.1


## 2. Проверка: сумма баллов == скор, посчитанный напрямую из PD

In [3]:
Xc = clip.transform(compute_features(raw))
n_chk = 3000
idx_chk = np.arange(n_chk)
tot = np.full(n_chk, base_pts)
for c in cols:
    bi = enc.bin_index(Xc.iloc[idx_chk], c)
    tot += np.array(card[c]["points"])[bi]
z = b0 + enc.transform(Xc.iloc[idx_chk])[cols].to_numpy() @ beta
pd_unclipped = 1 / (1 + np.exp(-(a * z + b)))
direct = BASE_SCORE - FACTOR * (logit(pd_unclipped) - logit(PD_REF))
print("макс. расхождение «сумма баллов» vs «напрямую из PD»:", float(np.abs(tot - direct).max()))
pd_model = model.predict_pd(raw.iloc[idx_chk])
print("PD модели совпадает с формулой (там, где не сработал клиппинг PD):", bool(np.allclose(pd_model[(pd_unclipped > 0.002) & (pd_unclipped < 0.95)], pd_unclipped[(pd_unclipped > 0.002) & (pd_unclipped < 0.95)])))

макс. расхождение «сумма баллов» vs «напрямую из PD»: 5.684341886080801e-13


PD модели совпадает с формулой (там, где не сработал клиппинг PD): True


## 3. Скор-карта в сохранённом виде + человекочитаемая таблица

In [4]:
json.dump(dict(version=model.version, pdo=PDO, base_score=BASE_SCORE, pd_ref=PD_REF, factor=FACTOR, base_points=float(base_pts),
               calibration_a=a, calibration_b=b, intercept=b0, score_min=300, score_max=900, features=card),
          open(MODELS / "scorecard_v1.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
lines = ["# Скор-карта (балльная модель)", "",
         "> Синтетические данные; скор-карта демонстрирует методологию и не переносится на реальный портфель.", "",
         f"Версия: `{model.version}`. Шкала: **{BASE_SCORE:.0f} баллов = PD {PD_REF:.0%}**, удвоение шансов каждые **{PDO:.0f} баллов**; баллы выше — риск ниже; итог ограничен диапазоном 300–900.",
         f"Базовая константа: **{base_pts:.1f}**. Итоговый балл = константа + сумма баллов по корзинам признаков.", "",
         "Колонка «вклад vs средний» = баллы корзины − средневзвешенные баллы признака по обучающей выборке (положительный — лучше среднего заёмщика).", ""]
for c in rng_pts.index:
    v = card[c]
    lines += [f"### {v['label']} (`{c}`, коэффициент {v['beta']:.3f})", "", "| Корзина | Заявок | Доля дефолтов | Баллы | Вклад vs средний |", "|---|---|---|---|---|"]
    for lab, n, br, p in zip(v["bins"], v["n"], v["bad_rate"], v["points"]):
        if n == 0 and lab == "__MISSING__": continue
        lab = "нет данных" if lab == "__MISSING__" else lab
        lines.append(f"| {lab} | {n} | {'—' if br is None else f'{br:.1%}'} | {p:+.1f} | {p - v['ref_points']:+.1f} |")
    lines.append("")
(ROOT / "docs/ml/scorecard.md").write_text("\n".join(lines), encoding="utf-8")
print("записано docs/ml/scorecard.md,", len(lines), "строк")
v = card["company_age_months"]; print(pd.DataFrame({"корзина": v["bins"], "баллы": np.round(v["points"], 1)}).to_string(index=False))

записано docs/ml/scorecard.md, 286 строк
       корзина  баллы
 (-inf; 8.968]   -7.2
(8.968; 15.97]   -5.2
(15.97; 35.94]   -3.2
(35.94; 57.92]    0.9
(57.92; 103.8]    3.3
 (103.8; +inf]   10.9
   __MISSING__    0.0


## 4. PD → баллы → рейтинговая шкала
Границы PD заданы заранее «круглыми» числами; проверяем, что в каждой градации достаточно наблюдений и фактическая доля дефолтов монотонно растёт.

In [5]:
pd_all = model.predict_pd(raw)
score_all = np.clip(np.round(BASE_SCORE - FACTOR * (logit(pd_all) - logit(PD_REF))), 300, 900)
print("квантили PD:", dict(zip([1, 5, 10, 25, 50, 75, 90, 95, 99], np.round(np.percentile(pd_all, [1, 5, 10, 25, 50, 75, 90, 95, 99]), 4))))
print("баллы: мин", score_all.min(), "медиана", np.median(score_all), "макс", score_all.max())

GRADES = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC", "CC", "C", "D"]
UPPER = [0.020, 0.030, 0.045, 0.065, 0.090, 0.125, 0.180, 0.280, 0.420, 1.0]    # верхние границы PD (не включая)
SEGMENT = dict(zip(GRADES, ["LOW"] * 3 + ["MEDIUM"] * 2 + ["HIGH"] * 2 + ["VERY_HIGH"] * 3))
def grade_of(p):
    return np.array(GRADES)[np.searchsorted(UPPER, p, side="right").clip(0, len(GRADES) - 1)]
g_all = grade_of(pd_all)
def rating_table(mask, name):
    d = pd.DataFrame({"g": g_all[mask], "pd": pd_all[mask], "y": y[mask]})
    t = d.groupby("g").agg(n=("y", "size"), mean_pd=("pd", "mean"), default_rate=("y", "mean")).reindex(GRADES)
    t.insert(0, "share", t.n / t.n.sum()); t.insert(0, "sample", name); return t
t_dev = rating_table(split != "test", "train+valid")      # границы подбирались на разработочной выборке; test трогаем только в ноутбуке 05
print(t_dev.round(4).to_string())
mono = lambda t: bool(t.default_rate.dropna().is_monotonic_increasing)
print("\nмонотонность фактической доли дефолтов (train+valid):", mono(t_dev))
print("мин. число наблюдений в градации:", int(t_dev.n.min()))

квантили PD: {1: np.float64(0.011), 5: np.float64(0.0145), 10: np.float64(0.0175), 25: np.float64(0.0245), 50: np.float64(0.0427), 75: np.float64(0.1126), 90: np.float64(0.2847), 95: np.float64(0.4181), 99: np.float64(0.6391)}
баллы: мин 300.0 медиана 612.0 макс 738.0
          sample   share     n  mean_pd  default_rate
g                                                    
AAA  train+valid  0.1506  1446   0.0157        0.0166
AA   train+valid  0.2045  1963   0.0247        0.0219
A    train+valid  0.1647  1581   0.0365        0.0354
BBB  train+valid  0.1071  1028   0.0536        0.0584
BB   train+valid  0.0761   731   0.0765        0.0739
B    train+valid  0.0680   653   0.1057        0.1210
CCC  train+valid  0.0564   541   0.1508        0.1682
CC   train+valid  0.0698   670   0.2250        0.2060
C    train+valid  0.0522   501   0.3430        0.3633
D    train+valid  0.0506   486   0.5462        0.5267

монотонность фактической доли дефолтов (train+valid): True
мин. число наблюдений в

## 5. Сохранение шкалы и документа `docs/ml/rating-scale.md`

In [6]:
scale = [dict(grade=g, segment=SEGMENT[g], pd_from=0.0 if i == 0 else UPPER[i - 1], pd_to=UPPER[i],
              score_from=int(np.clip(np.round(BASE_SCORE - FACTOR * (logit(UPPER[i] - 1e-9 if i < 9 else 0.9999) - logit(PD_REF))), 300, 900)),
              score_to=int(np.clip(np.round(BASE_SCORE - FACTOR * (logit(max(0.002, 0.0 if i == 0 else UPPER[i - 1])) - logit(PD_REF))), 300, 900)))
         for i, g in enumerate(GRADES)]
json.dump(dict(version="rating-scale-v1", grades=scale), open(MODELS / "rating_scale_v1.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
seg_ru = {"LOW": "низкий", "MEDIUM": "средний", "HIGH": "повышенный", "VERY_HIGH": "высокий"}
L = ["# Рейтинговая шкала", "",
     "> Синтетические данные; шкала демонстрирует методологию. Пороги одобрения и бизнес-правила банка в шкалу **не** заложены — они живут в `credit-core`.", "",
     f"Шкала из **{len(GRADES)} градаций** (9 по PD + `D` — самая рискованная). PD — калиброванная вероятность дефолта в течение 12 месяцев. Баллы: 600 = PD 5%, +50 баллов = удвоение шансов (диапазон 300–900).", "",
     "| Рейтинг | Сегмент риска | PD от | PD до | Баллы | Доля заявок | Средний PD | Факт. дефолты | Наблюдений |", "|---|---|---|---|---|---|---|---|---|"]
for s, g in zip(scale, GRADES):
    r1 = t_dev.loc[g]
    L.append(f"| **{g}** | {seg_ru[s['segment']]} | {s['pd_from']:.1%} | {s['pd_to']:.1%} | {s['score_to']}–{s['score_from']} | {r1.share:.1%} | {r1.mean_pd:.2%} | {r1.default_rate:.2%} | {int(r1.n)} |")
L += ["", f"Статистика — на разработочной выборке (train+valid, границы подбирались на ней); монотонность фактической доли дефолтов: {mono(t_dev)}. Итоговая таблица на отложенной тестовой выборке — в `model-report.md`.", ""]
(ROOT / "docs/ml/rating-scale.md").write_text("\n".join(L), encoding="utf-8")
print("\n".join(L[6:18]))

| Рейтинг | Сегмент риска | PD от | PD до | Баллы | Доля заявок | Средний PD | Факт. дефолты | Наблюдений |
|---|---|---|---|---|---|---|---|---|
| **AAA** | низкий | 0.0% | 2.0% | 836–668 | 15.1% | 1.57% | 1.66% | 1446 |
| **AA** | низкий | 2.0% | 3.0% | 668–638 | 20.4% | 2.47% | 2.19% | 1963 |
| **A** | низкий | 3.0% | 4.5% | 638–608 | 16.5% | 3.65% | 3.54% | 1581 |
| **BBB** | средний | 4.5% | 6.5% | 608–580 | 10.7% | 5.36% | 5.84% | 1028 |
| **BB** | средний | 6.5% | 9.0% | 580–554 | 7.6% | 7.65% | 7.39% | 731 |
| **B** | повышенный | 9.0% | 12.5% | 554–528 | 6.8% | 10.57% | 12.10% | 653 |
| **CCC** | повышенный | 12.5% | 18.0% | 528–497 | 5.6% | 15.08% | 16.82% | 541 |
| **CC** | высокий | 18.0% | 28.0% | 497–456 | 7.0% | 22.50% | 20.60% | 670 |
| **C** | высокий | 28.0% | 42.0% | 456–411 | 5.2% | 34.30% | 36.33% | 501 |
| **D** | высокий | 42.0% | 100.0% | 411–300 | 5.1% | 54.62% | 52.67% | 486 |


## 6. Графики: распределение баллов и доля дефолтов по рейтингам

In [7]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for lab, col, m in (("не дефолт", "#4C78A8", y == 0), ("дефолт", "#E45756", y == 1)):
    ax[0].hist(score_all[m & (split != "test")], bins=40, alpha=.65, color=col, label=lab, density=True)
ax[0].set_xlabel("скор (баллы, выше — надёжнее)"); ax[0].set_ylabel("плотность"); ax[0].set_title("Распределение скора, train+valid"); ax[0].legend()
x = np.arange(len(GRADES))
ax[1].bar(x - .2, t_dev.default_rate * 100, .4, label="факт (train+valid)", color="#4C78A8")
ax[1].bar(x + .2, t_dev.mean_pd * 100, .4, label="средний PD", color="#F58518")
ax[1].set_xticks(x); ax[1].set_xticklabels(GRADES); ax[1].set_yscale("log"); ax[1].set_ylabel("%, лог. шкала"); ax[1].set_title("Рейтинги: PD vs факт"); ax[1].legend()
plt.tight_layout(); plt.savefig(IMG / "04_score_distribution_ratings.png", dpi=130); plt.close()
print("сохранено 04_score_distribution_ratings.png")

сохранено 04_score_distribution_ratings.png
